<a href="https://colab.research.google.com/github/emdt-cyber/test-capstone/blob/main/test_sudhra_hua.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**README.md**

In [51]:
!git clone https://github.com/emdt-cyber/delhivery-delay-pipeline.git

Cloning into 'delhivery-delay-pipeline'...
fatal: could not read Username for 'https://github.com': No such device or address


In [52]:
%cd delhivery-delay-pipeline

[Errno 2] No such file or directory: 'delhivery-delay-pipeline'
/content


In [53]:
!mkdir -p data src models outputs

In [54]:
!pwd
!ls

/content
data  delhivery-delay-pipeline-v2  models  outputs  sample_data  src


In [55]:
!git clone https://github.com/emdt-cyber/delhivery-delay-pipeline.git

Cloning into 'delhivery-delay-pipeline'...
fatal: could not read Username for 'https://github.com': No such device or address


In [56]:
%cd delhivery-delay-pipeline

[Errno 2] No such file or directory: 'delhivery-delay-pipeline'
/content


**requirements.txt**

In [57]:
import pandas as pd
import numpy as np
import sklearn
import joblib

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Joblib:", joblib.__version__)

Pandas: 2.2.3
NumPy: 2.1.3
Scikit-learn: 1.6.1
Joblib: 1.6.0


**generate_data.py**

In [58]:
"""Synthetic shipment data for the Delay Intelligence capstone.
Run once:  python generate_data.py   (writes data/hubs.csv, data/raw_shipments.csv, data/new_shipments.csv)
Do not edit any constant: every number in the brief was measured on this exact output."""
import os
import numpy as np
import pandas as pd

HUBS = [
    ("HUB01", "Mumbai", "West", 1, 1500), ("HUB02", "Delhi", "North", 1, 1600),
    ("HUB03", "Bengaluru", "South", 1, 1450), ("HUB04", "Hyderabad", "South", 1, 1350),
    ("HUB05", "Chennai", "South", 1, 1300), ("HUB06", "Pune", "West", 1, 1250),
    ("HUB07", "Kolkata", "East", 1, 1200), ("HUB08", "Ahmedabad", "West", 1, 1150),
    ("HUB09", "Jaipur", "North", 2, 950), ("HUB10", "Lucknow", "North", 2, 900),
    ("HUB11", "Chandigarh", "North", 2, 850), ("HUB12", "Indore", "West", 2, 820),
    ("HUB13", "Nagpur", "West", 2, 780), ("HUB14", "Coimbatore", "South", 2, 740),
    ("HUB15", "Kochi", "South", 2, 700), ("HUB16", "Surat", "West", 2, 650),
    ("HUB17", "Patna", "East", 3, 520), ("HUB18", "Ranchi", "East", 3, 480),
    ("HUB19", "Guwahati", "East", 3, 450), ("HUB20", "Raipur", "East", 3, 420),
    ("HUB21", "Bhubaneswar", "East", 3, 390), ("HUB22", "Dehradun", "North", 3, 350),
    ("HUB23", "Siliguri", "East", 3, 310), ("HUB24", "Agartala", "East", 3, 270),
]
DIST_MU = [180, 320, 480]      # mean lane distance (km) for hub tier 1 / 2 / 3
DIST_SD = [50, 90, 150]
STOPS_MU = [55, 70, 85]        # mean stops on the rider route for hub tier 1 / 2 / 3
HUB_EFFECT = [0, 2, 4]         # extra hours for hub tier 1 / 2 / 3
# How a hub really operates (1 = metro-like, 2 = regional, 3 = stretched). Four hubs operate unlike their tier label.
OPERATING_TIER = {"HUB07": 2, "HUB13": 3, "HUB15": 3, "HUB22": 2}
BASE_HOURS = {"Express": 10, "Standard": 28, "Economy": 58}


def simulate(n_per_hub, seed):
    """Clean shipments, n_per_hub rows for each of the 24 hubs, in hub order."""
    np.random.seed(seed)
    hubs = pd.DataFrame(HUBS, columns=["hub_id", "hub_city", "region", "hub_tier", "hub_daily_capacity"])
    n = n_per_hub * len(hubs)
    op_tier = [OPERATING_TIER.get(h, t) for h, t in zip(hubs["hub_id"], hubs["hub_tier"])]
    tier = np.repeat(op_tier, n_per_hub) - 1
    hub_id = np.repeat(hubs["hub_id"].values, n_per_hub)
    distance = np.clip(np.random.normal(np.array(DIST_MU)[tier], np.array(DIST_SD)[tier]), 15, 1500).round(1)
    stops = np.clip(np.random.normal(np.array(STOPS_MU)[tier], 10), 20, 140).round().astype(int)
    weight = np.clip(np.random.lognormal(0.6, 0.7, n), 0.1, 40).round(2)
    value = np.random.lognormal(7.0, 0.8, n).round().astype(int)
    service = np.random.choice(["Express", "Standard", "Economy"], n, p=[0.2, 0.55, 0.25])
    pay = np.random.choice(["Prepaid", "COD"], n, p=[0.6, 0.4])
    rain = (np.random.rand(n) < 0.18).astype(int)
    day = np.random.randint(0, 90, n)
    hour = np.random.randint(6, 22, n)
    minute = np.random.randint(0, 60, n)
    ts = pd.Timestamp("2026-07-01") + pd.to_timedelta(day, unit="D") + pd.to_timedelta(hour * 60 + minute, unit="m")
    sale = ((ts >= pd.Timestamp("2026-09-15")) & (ts < pd.Timestamp("2026-09-29"))).astype(int)
    weekend = (ts.dayofweek >= 5).astype(int)
    hours = (np.array([BASE_HOURS[s] for s in service]) + 0.03 * distance + 0.10 * stops
             + np.array(HUB_EFFECT)[tier] + 0.12 * weight + 4 * rain + 7 * sale
             + 3 * (pay == "COD") + 2 * weekend + 3 * (hour >= 18)
             + 12 * sale * (service == "Express") + 8 * (distance > 400) * rain
             + np.random.normal(0, 4, n)).round(2)
    return pd.DataFrame({
        "hub_id": hub_id, "service_type": service, "payment_mode": pay,
        "parcel_weight_kg": weight, "distance_km": distance, "declared_value": value,
        "n_stops": stops, "rain_flag": rain, "sale_event_flag": sale, "dispatch_time": ts,
        "label_print_batch": np.random.randint(1000, 9999, n),
        "scanner_build_no": np.random.randint(1, 40, n),
        "tape_roll_no": np.random.randint(1, 500, n),
        "sticker_serial_mod": np.random.uniform(0, 1, n).round(3),
        "shelf_slot_no": np.random.randint(1, 300, n),
        "delivery_hours": hours,
    })


def make_messy(df, seed, n_missing, n_case, n_sentinel, n_dupes, id_start):
    """Turn a clean table into a raw export: text columns, gaps, duplicates, a sentinel."""
    np.random.seed(seed)
    df = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    n = len(df)
    df.insert(0, "shipment_id", ["SHP" + str(id_start + i) for i in range(n)])
    df["dispatch_time"] = df["dispatch_time"].dt.strftime("%Y-%m-%d %H:%M")
    df["declared_value"] = df["declared_value"].map(lambda v: "\u20b9" + format(v, ","))
    for col, k in [("distance_km", n_missing[0]), ("parcel_weight_kg", n_missing[1]),
                   ("declared_value", n_missing[2]), ("payment_mode", n_missing[3])]:
        df.loc[np.random.choice(n, k, replace=False), col] = np.nan
    pick = np.random.choice(n, n_case, replace=False)
    shapes = np.random.randint(0, 3, n_case)
    for i, s in zip(pick, shapes):
        v = df.at[i, "service_type"]
        df.at[i, "service_type"] = [v.upper(), v.lower(), "  " + v + " "][s]
    if n_sentinel:
        df.loc[np.random.choice(n, n_sentinel, replace=False), "parcel_weight_kg"] = 999.0
    if n_dupes:
        df = pd.concat([df, df.iloc[np.random.choice(n, n_dupes, replace=False)]], ignore_index=True)
    return df


if __name__ == "__main__":
    os.makedirs("data", exist_ok=True)
    hubs = pd.DataFrame(HUBS, columns=["hub_id", "hub_city", "region", "hub_tier", "hub_daily_capacity"])
    hubs.to_csv("data/hubs.csv", index=False)
    raw = make_messy(simulate(50, 2604), seed=7, n_missing=(36, 24, 12, 18),
                     n_case=120, n_sentinel=5, n_dupes=24, id_start=10001)
    raw.to_csv("data/raw_shipments.csv", index=False)
    new = make_messy(simulate(2, 99), seed=8, n_missing=(3, 2, 1, 2),
                     n_case=6, n_sentinel=0, n_dupes=0, id_start=90001)
    new = new.drop(columns=["delivery_hours"])
    new.to_csv("data/new_shipments.csv", index=False)
    print(len(hubs), len(raw), len(new))

24 1224 48


**shipment_prep.py**

In [59]:
"""Shared shipment cleaning and feature engineering for the capstone."""
import numpy as np
import pandas as pd

PROMISED_HOURS = {"Express": 36, "Standard": 60, "Economy": 96}

FEATURES = [
    "promised_hours", "distance_km", "parcel_weight_kg", "log_declared_value",
    "n_stops", "rain_flag", "sale_event_flag", "is_cod", "is_weekend",
    "dispatch_hour", "hub_tier", "hub_daily_capacity",
    "label_print_batch", "scanner_build_no", "tape_roll_no",
    "sticker_serial_mod", "shelf_slot_no",
]

NOISE_COLUMNS = [
    "label_print_batch", "scanner_build_no", "tape_roll_no",
    "sticker_serial_mod", "shelf_slot_no",
]


def clean_shipments(raw, medians=None):
    """Return (clean_df, medians) after applying the specified cleaning steps."""
    df = raw.drop_duplicates().copy()
    df["service_type"] = df["service_type"].astype(str).str.strip().str.title()
    df["payment_mode"] = df["payment_mode"].fillna("Unknown")
    df["declared_value_inr"] = (
        df["declared_value"].astype(str)
        .str.replace("₹", "", regex=False)
        .str.replace(",", "", regex=False)
        .replace({"nan": np.nan, "None": np.nan, "": np.nan})
        .astype(float)
    )
    df = df.drop(columns=["declared_value"])
    df["dispatch_time"] = pd.to_datetime(df["dispatch_time"])
    df.loc[df["parcel_weight_kg"] > 100, "parcel_weight_kg"] = np.nan

    if medians is None:
        medians = {
            "distance_km": float(df["distance_km"].median()),
            "parcel_weight_kg": float(df["parcel_weight_kg"].median()),
            "declared_value_inr": float(df["declared_value_inr"].median()),
        }
    else:
        required = {"distance_km", "parcel_weight_kg", "declared_value_inr"}
        if set(medians) != required:
            raise ValueError(f"medians must contain exactly these keys: {sorted(required)}")
        medians = {k: float(v) for k, v in medians.items()}

    for column in ("distance_km", "parcel_weight_kg", "declared_value_inr"):
        df[column] = df[column].fillna(medians[column])
    return df, medians


def build_features(clean, hubs):
    """Attach hub information and create features; does not use delivery_hours."""
    df = clean.merge(hubs, on="hub_id", how="left", validate="many_to_one")
    if df[["hub_city", "region", "hub_tier", "hub_daily_capacity"]].isna().any().any():
        raise ValueError("One or more shipment hub_ids did not match hubs.csv")
    df["promised_hours"] = df["service_type"].map(PROMISED_HOURS).astype(int)
    df["is_cod"] = (df["payment_mode"] == "COD").astype(int)
    df["is_weekend"] = (df["dispatch_time"].dt.dayofweek >= 5).astype(int)
    df["dispatch_hour"] = df["dispatch_time"].dt.hour.astype(int)
    df["log_declared_value"] = np.log(df["declared_value_inr"].astype(float))
    return df

**part_a_clean_eda.py**

In [60]:
"""Part A: clean shipments, build the modelling table and produce evidence CSVs."""
import os
import numpy as np
import pandas as pd
from src.shipment_prep import clean_shipments, build_features, FEATURES

os.makedirs("data", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
raw = pd.read_csv("data/raw_shipments.csv")
hubs = pd.read_csv("data/hubs.csv")

# Counts before cleaning, with duplicate removal applied as required.
raw_rows = len(raw)
dedup = raw.drop_duplicates().copy()
rows_after_dedup = len(dedup)
duplicate_rows_removed = raw_rows - rows_after_dedup
missing_distance_before = int(dedup["distance_km"].isna().sum())
missing_parcel_weight_before = int(dedup["parcel_weight_kg"].isna().sum())
missing_declared_value_before = int(dedup["declared_value"].isna().sum())
missing_payment_mode_before = int(dedup["payment_mode"].isna().sum())
sentinel_weight_rows = int((dedup["parcel_weight_kg"] > 100).sum())
service_type_spellings_before = int(dedup["service_type"].nunique())
service_type_spellings_after = int(dedup["service_type"].astype(str).str.strip().str.title().nunique())

clean, medians = clean_shipments(raw)
full = build_features(clean, hubs)
full["delivery_hours"] = pd.to_numeric(full["delivery_hours"], errors="raise")
full["is_delayed"] = (full["delivery_hours"] > full["promised_hours"]).astype(int)

# Confirm the left merge did not add or lose shipment rows.
rows_after_hub_merge = len(full)
full.to_csv("data/modelling_table.csv", index=False)

# Distribution and outlier checks. Bounds use unrounded quartiles.
q1 = float(full["delivery_hours"].quantile(0.25))
q3 = float(full["delivery_hours"].quantile(0.75))
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
hours_outlier_count = int(((full["delivery_hours"] < lower_bound) | (full["delivery_hours"] > upper_bound)).sum())

nulls_in_features_and_target = int(full[FEATURES + ["delivery_hours", "is_delayed"]].isna().sum().sum())
filled_log = np.log(full["declared_value_inr"].astype(float))
checks = [
    ("raw_rows", raw_rows),
    ("duplicate_rows_removed", duplicate_rows_removed),
    ("rows_after_dedup", rows_after_dedup),
    ("missing_distance_km_before", missing_distance_before),
    ("missing_parcel_weight_before", missing_parcel_weight_before),
    ("sentinel_weight_rows", sentinel_weight_rows),
    ("missing_declared_value_before", missing_declared_value_before),
    ("missing_payment_mode_before", missing_payment_mode_before),
    ("service_type_spellings_before", service_type_spellings_before),
    ("service_type_spellings_after", service_type_spellings_after),
    ("median_distance_km", medians["distance_km"]),
    ("median_parcel_weight_kg", medians["parcel_weight_kg"]),
    ("median_declared_value_inr", medians["declared_value_inr"]),
    ("rows_after_hub_merge", rows_after_hub_merge),
    ("nulls_in_features_and_target", nulls_in_features_and_target),
    ("delayed_count", int(full["is_delayed"].sum())),
    ("delay_rate", float(full["is_delayed"].mean())),
    ("hours_q1", q1), ("hours_q3", q3),
    ("hours_lower_bound", lower_bound), ("hours_upper_bound", upper_bound),
    ("hours_outlier_count", hours_outlier_count),
    ("declared_value_mean_inr", float(full["declared_value_inr"].mean())),
    ("declared_value_median_inr", float(full["declared_value_inr"].median())),
    ("log_declared_value_mean", float(filled_log.mean())),
    ("log_declared_value_median", float(filled_log.median())),
]
pd.DataFrame(checks, columns=["metric", "value"]).to_csv("outputs/part_a_checks.csv", index=False, float_format="%.6f")

# Group delay evidence.
rows = []
for dimension in ["service_type", "hub_tier"]:
    for group, g in full.groupby(dimension, sort=True):
        rows.append({"dimension": dimension, "group": group, "shipments": len(g),
                     "delayed": int(g["is_delayed"].sum()), "delay_rate": g["is_delayed"].mean(),
                     "mean_hours": g["delivery_hours"].mean()})
full["distance_band"] = pd.cut(
    full["distance_km"], bins=[0, 200, 400, 600, 2000],
    labels=["0-200", "201-400", "401-600", "601+"], right=True, include_lowest=True
)
for group, g in full.groupby("distance_band", observed=False, sort=True):
    if len(g):
        rows.append({"dimension": "distance_band", "group": str(group), "shipments": len(g),
                     "delayed": int(g["is_delayed"].sum()), "delay_rate": g["is_delayed"].mean(),
                     "mean_hours": g["delivery_hours"].mean()})
pd.DataFrame(rows, columns=["dimension", "group", "shipments", "delayed", "delay_rate", "mean_hours"]).to_csv(
    "outputs/part_a_delay_by_group.csv", index=False, float_format="%.6f")

full.select_dtypes(include=[np.number]).corr().to_csv("outputs/part_a_correlations.csv", index_label="column", float_format="%.6f")

if rows_after_hub_merge != rows_after_dedup:
    raise AssertionError("Hub merge changed the deduplicated shipment row count")
if nulls_in_features_and_target != 0:
    raise AssertionError("Missing values remain in the modelling features or target")
print("Part A complete: data/modelling_table.csv and Part A outputs written.")
print("Cleaning medians:", medians)

ModuleNotFoundError: No module named 'src.shipment_prep'

**part_b_regression.py**

In [61]:
"""Part B: leakage-safe regression experiments for delivery_hours."""
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from src.shipment_prep import FEATURES

os.makedirs("outputs", exist_ok=True)
df = pd.read_csv("data/modelling_table.csv")
X = df[FEATURES].astype(float)
y = df["delivery_hours"].astype(float)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

scaler = StandardScaler().fit(X_train)
train_scaled = scaler.transform(X_train)
test_scaled = scaler.transform(X_test)
checks = [
    ("train_rows", len(X_train)), ("val_rows", len(X_val)), ("test_rows", len(X_test)),
    ("train_scaled_max_abs_mean", float(np.abs(train_scaled.mean(axis=0)).max())),
    ("test_scaled_max_abs_mean", float(np.abs(test_scaled.mean(axis=0)).max())),
]
pd.DataFrame(checks, columns=["metric", "value"]).to_csv("outputs/part_b_checks.csv", index=False, float_format="%.10f")

# Tree-depth diagnostic: fit each depth only on training rows.
depths = [1, 2, 3, 4, 6, 8, 12, None]
tree_rows = []
for depth in depths:
    tree = DecisionTreeRegressor(max_depth=depth, random_state=42).fit(X_train, y_train)
    train_pred, val_pred = tree.predict(X_train), tree.predict(X_val)
    tree_rows.append({"max_depth": "unlimited" if depth is None else depth,
                      "train_rmse": mean_squared_error(y_train, train_pred) ** 0.5,
                      "val_rmse": mean_squared_error(y_val, val_pred) ** 0.5})
tree_table = pd.DataFrame(tree_rows)
tree_table.to_csv("outputs/part_b_tree_depth.csv", index=False, float_format="%.6f")
tree_table["depth_order"] = tree_table["max_depth"].map(
    lambda value: 999 if str(value) == "unlimited" else int(value)
)
best_depth_row = tree_table.sort_values(["val_rmse", "depth_order"]).iloc[0]
best_depth = None if str(best_depth_row["max_depth"]) == "unlimited" else int(best_depth_row["max_depth"])
tree_table = tree_table.drop(columns=["depth_order"])

models = {
    "linear_regression": Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())]),
    "decision_tree": DecisionTreeRegressor(max_depth=best_depth, random_state=42),
    "random_forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "gradient_boosting": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
}

def metric_row(name, split, truth, pred):
    mse = mean_squared_error(truth, pred)
    return {"model": name, "split": split, "MAE": mean_absolute_error(truth, pred),
            "MSE": mse, "RMSE": float(np.sqrt(mse)), "R2": r2_score(truth, pred)}

baseline_pred = np.full(len(y_val), float(y_train.mean()))
metric_rows = [metric_row("mean_baseline", "val", y_val, baseline_pred)]
fitted = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted[name] = model
    metric_rows.append(metric_row(name, "val", y_val, model.predict(X_val)))

# Learning-rate experiment.
gb_rows = []
for lr in [0.01, 0.1, 1.0]:
    gb = GradientBoostingRegressor(n_estimators=100, learning_rate=lr, random_state=42).fit(X_train, y_train)
    pred = gb.predict(X_val)
    gb_rows.append({"learning_rate": lr, "val_rmse": mean_squared_error(y_val, pred) ** 0.5,
                    "val_r2": r2_score(y_val, pred)})
pd.DataFrame(gb_rows).to_csv("outputs/part_b_gb_learning_rate.csv", index=False, float_format="%.6f")

# Compare the two best validation candidates on the 1,020 train+validation rows.
ranked = sorted(models, key=lambda name: mean_squared_error(y_val, fitted[name].predict(X_val)) ** 0.5)
X_tv = pd.concat([X_train, X_val], axis=0)
y_tv = pd.concat([y_train, y_val], axis=0)
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
cv_rows = []
for name in ranked[:2]:
    scores = cross_val_score(models[name], X_tv, y_tv, cv=kfold)
    cv_rows.append({"model": name, "mean_r2": scores.mean(), "std_r2": scores.std()})
pd.DataFrame(cv_rows).to_csv("outputs/part_b_cv.csv", index=False, float_format="%.6f")

# Only the validation-RMSE winner is evaluated on the test rows, once.
final_name = ranked[0]
metric_rows.append(metric_row(final_name, "test", y_test, fitted[final_name].predict(X_test)))
pd.DataFrame(metric_rows, columns=["model", "split", "MAE", "MSE", "RMSE", "R2"]).to_csv(
    "outputs/part_b_metrics.csv", index=False, float_format="%.6f")
print("Part B complete. Decision-tree depth:", best_depth, "Final candidate:", final_name)

ModuleNotFoundError: No module named 'src.shipment_prep'

**part_c_classification.py**

In [62]:
"""Part C: delay classification, feature evidence, tuning and alert threshold."""
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score)
from src.shipment_prep import FEATURES, NOISE_COLUMNS

os.makedirs("outputs", exist_ok=True)
df = pd.read_csv("data/modelling_table.csv")
X = df[FEATURES].astype(float)
y = df["is_delayed"].astype(int)
X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.5, random_state=42, stratify=y_tv)


def metrics(name, truth, pred):
    tn, fp, fn, tp = confusion_matrix(truth, pred, labels=[0, 1]).ravel()
    return {"model": name, "TN": tn, "FP": fp, "FN": fn, "TP": tp,
            "accuracy": accuracy_score(truth, pred),
            "precision": precision_score(truth, pred, zero_division=0),
            "recall": recall_score(truth, pred, zero_division=0),
            "f1": f1_score(truth, pred, zero_division=0)}

# Baseline and three validation classifiers.
model_rows = [metrics("blind_all_on_time", y_val, np.zeros(len(y_val), dtype=int))]
log_model = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression())]).fit(X_train, y_train)
model_rows.append(metrics("logistic_regression", y_val, log_model.predict(X_val)))

tree_candidates = []
for depth in [2, 3, 4, 5, 6, 8, 10]:
    tree = DecisionTreeClassifier(criterion="gini", max_depth=depth, random_state=42).fit(X_train, y_train)
    pred = tree.predict(X_val)
    score = f1_score(y_val, pred, zero_division=0)
    tree_candidates.append((score, -depth, depth, tree, pred))
_, _, best_tree_depth, best_tree, best_tree_pred = max(tree_candidates, key=lambda x: (x[0], x[1]))
model_rows.append(metrics(f"decision_tree_depth_{best_tree_depth}", y_val, best_tree_pred))
rf_default_17 = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
model_rows.append(metrics("random_forest_default", y_val, rf_default_17.predict(X_val)))
pd.DataFrame(model_rows).to_csv("outputs/part_c_models.csv", index=False, float_format="%.6f")

# Feature F-scores and forest importance are fit on training rows only.
forest_for_importance = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
selector_all = SelectKBest(f_classif, k="all").fit(X_train, y_train)
scores = selector_all.scores_
order = np.argsort(-scores, kind="stable")
ranks = np.empty(len(scores), dtype=int)
ranks[order] = np.arange(1, len(scores) + 1)
feature_table = pd.DataFrame({"feature": FEATURES, "f_score": scores, "f_rank": ranks,
                              "rf_importance": forest_for_importance.feature_importances_,
                              "is_noise": [f in NOISE_COLUMNS for f in FEATURES]})
feature_table.to_csv("outputs/part_c_feature_scores.csv", index=False, float_format="%.6f")

# Validation F1 across the requested selector sizes.
selectk_rows = []
for k in [5, 7, 10, 12, 17]:
    pipe = Pipeline([("selector", SelectKBest(f_classif, k=k)),
                     ("model", RandomForestClassifier(n_estimators=100, random_state=42))])
    pipe.fit(X_train, y_train)
    selectk_rows.append({"k": k, "val_f1": f1_score(y_val, pipe.predict(X_val), zero_division=0)})
pd.DataFrame(selectk_rows).to_csv("outputs/part_c_selectk.csv", index=False, float_format="%.6f")

# Fit selector on training rows only; transform validation with the same selector.
selector7 = SelectKBest(f_classif, k=7).fit(X_train, y_train)
X_train7 = selector7.transform(X_train)
X_val7 = selector7.transform(X_val)
param_grid = {"n_estimators": [50, 100, 150], "max_depth": [4, 6, 8], "min_samples_leaf": [1, 5, 10]}
grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid=param_grid, cv=5)
grid.fit(X_train7, y_train)
random_search = RandomizedSearchCV(RandomForestClassifier(random_state=42), param_distributions=param_grid,
                                   n_iter=9, cv=5, random_state=42)
random_search.fit(X_train7, y_train)
deforest = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=None, min_samples_leaf=1)
deforest.fit(X_train7, y_train)

tuning_rows = []
for name, candidates, folds, fits, params, best_acc, model in [
    ("grid_search", 27, 5, 135, grid.best_params_, grid.best_score_, grid.best_estimator_),
    ("randomized_search", 9, 5, 45, random_search.best_params_, random_search.best_score_, random_search.best_estimator_),
    ("default_forest_same_7_features", 1, 0, 1,
     {"n_estimators": 100, "max_depth": "unlimited", "min_samples_leaf": 1}, np.nan, deforest),
]:
    tuning_rows.append({"search": name, "candidates": candidates, "cv_folds": folds, "total_fits": fits,
                        "best_params": str(params), "best_cv_accuracy": best_acc,
                        "val_f1": f1_score(y_val, model.predict(X_val7), zero_division=0)})
pd.DataFrame(tuning_rows).to_csv("outputs/part_c_tuning.csv", index=False, float_format="%.6f")

# Choose highest validation F1; ties: default, then grid, then randomized.
priority = {"default_forest_same_7_features": 0, "grid_search": 1, "randomized_search": 2}
tuning_df = pd.DataFrame(tuning_rows)
chosen = tuning_df.sort_values(["val_f1"], ascending=False).copy()
max_f1 = chosen["val_f1"].max()
chosen = chosen[np.isclose(chosen["val_f1"], max_f1)]
chosen_name = min(chosen["search"], key=lambda name: priority[name])
chosen_params = {
    "grid_search": grid.best_params_,
    "randomized_search": random_search.best_params_,
    "default_forest_same_7_features": {"n_estimators": 100, "max_depth": None, "min_samples_leaf": 1},
}[chosen_name]

final_pipe = Pipeline([("selector", SelectKBest(f_classif, k=7)),
                       ("model", RandomForestClassifier(random_state=42, **chosen_params))])
final_pipe.fit(X_train, y_train)
val_prob = final_pipe.predict_proba(X_val)[:, 1]
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
threshold_rows = []
for threshold in thresholds:
    threshold_rows.append({"threshold": threshold, **{k: v for k, v in metrics("threshold", y_val,
                        (val_prob >= threshold).astype(int)).items() if k != "model"}})
threshold_table = pd.DataFrame(threshold_rows)
threshold_table.to_csv("outputs/part_c_thresholds.csv", index=False, float_format="%.6f")
eligible = threshold_table[threshold_table["recall"] >= 0.80]
chosen_threshold = float(eligible["threshold"].max()) if len(eligible) else 0.1

# Test rows are evaluated once, after the final model and threshold are fixed.
test_prob = final_pipe.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= chosen_threshold).astype(int)
test_metrics = metrics("final_test", y_test, test_pred)
pd.DataFrame([{k: test_metrics[k] for k in ["TN", "FP", "FN", "TP", "accuracy", "precision", "recall", "f1"]} | {"threshold": chosen_threshold}]).loc[:,
    ["threshold", "TN", "FP", "FN", "TP", "accuracy", "precision", "recall", "f1"]].to_csv(
    "outputs/part_c_final_test.csv", index=False, float_format="%.6f")

max_depth_value = chosen_params.get("max_depth", None)
pd.DataFrame([{"selector_k": 7, "n_estimators": chosen_params["n_estimators"],
               "max_depth": "unlimited" if max_depth_value is None else max_depth_value,
               "min_samples_leaf": chosen_params["min_samples_leaf"],
               "threshold": chosen_threshold, "chosen_from": chosen_name}]).to_csv(
    "outputs/part_c_final_config.csv", index=False)
print("Part C complete. Selected:", chosen_name, "threshold:", chosen_threshold)

ModuleNotFoundError: No module named 'src.shipment_prep'

**part_d_segments.py**

In [63]:
"""Part D: hub profiles, fixed K=3 segments and tier mismatch evidence."""
import os
from itertools import combinations
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

os.makedirs("outputs", exist_ok=True)
df = pd.read_csv("data/modelling_table.csv")
hubs = pd.read_csv("data/hubs.csv")
profile = (df.groupby("hub_id", sort=True)
           .agg(shipments=("is_delayed", "size"),
                mean_delivery_hours=("delivery_hours", "mean"),
                delay_rate=("is_delayed", "mean"),
                mean_distance_km=("distance_km", "mean"),
                mean_n_stops=("n_stops", "mean"))
           .reset_index().sort_values("hub_id").reset_index(drop=True))
profile.to_csv("outputs/hub_profile.csv", index=False, float_format="%.6f")

profile_columns = ["mean_delivery_hours", "delay_rate", "mean_distance_km", "mean_n_stops"]
X_scaled = StandardScaler().fit_transform(profile[profile_columns])


def across_over_within(X, labels):
    within, across = [], []
    for i, j in combinations(range(len(X)), 2):
        distance = float(np.linalg.norm(X[i] - X[j]))
        (within if labels[i] == labels[j] else across).append(distance)
    return float(np.mean(across) / np.mean(within))

separation_rows = []
for k in [2, 3, 4, 5]:
    model = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
    sizes = sorted(np.bincount(model.labels_).tolist(), reverse=True)
    separation_rows.append({"K": k, "across_over_within": across_over_within(X_scaled, model.labels_),
                            "cluster_sizes": "-".join(map(str, sizes))})
pd.DataFrame(separation_rows).to_csv("outputs/part_d_separation.csv", index=False, float_format="%.6f")

kmeans3 = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_scaled)
profile["cluster"] = kmeans3.labels_
cluster_order = profile.groupby("cluster")["delay_rate"].mean().sort_values().index.tolist()
segment_names = ["Standard", "Watch", "Escalate"]
cluster_to_segment = dict(zip(cluster_order, segment_names))
profile["segment"] = profile["cluster"].map(cluster_to_segment)

hub_segments = profile[["hub_id", "segment"]].merge(
    hubs[["hub_id", "hub_city", "hub_tier"]], on="hub_id", how="left", validate="one_to_one")
hub_segments = hub_segments[["hub_id", "segment", "hub_city", "hub_tier"]]
hub_segments.to_csv("outputs/hub_segments.csv", index=False)

segment_profile = (profile.groupby("segment")
                   .agg(hubs=("hub_id", "size"),
                        mean_delivery_hours=("mean_delivery_hours", "mean"),
                        delay_rate=("delay_rate", "mean"),
                        mean_distance_km=("mean_distance_km", "mean"),
                        mean_n_stops=("mean_n_stops", "mean"))
                   .reindex(segment_names).reset_index())
segment_profile.to_csv("outputs/segment_profile.csv", index=False, float_format="%.6f")

tier_expected = {1: "Standard", 2: "Watch", 3: "Escalate"}
hub_segments["expected_segment"] = hub_segments["hub_tier"].map(tier_expected)
mismatch = hub_segments[hub_segments["segment"] != hub_segments["expected_segment"]]
mismatch[["hub_id", "segment", "hub_city", "hub_tier"]].to_csv(
    "outputs/tier_vs_segment_mismatch.csv", index=False)
print("Part D segmentation complete.")

FileNotFoundError: [Errno 2] No such file or directory: 'data/modelling_table.csv'

**package_model.py**

In [64]:
"""Rebuild Part C's final pipeline and persist the reusable model artefacts."""
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from src.shipment_prep import FEATURES, clean_shipments, build_features

os.makedirs("models", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
raw = pd.read_csv("data/raw_shipments.csv")
hubs = pd.read_csv("data/hubs.csv")
clean, medians = clean_shipments(raw)
full = build_features(clean, hubs)
full["is_delayed"] = (full["delivery_hours"] > full["promised_hours"]).astype(int)
X = full[FEATURES].astype(float)
y = full["is_delayed"].astype(int)
X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.5, random_state=42, stratify=y_tv)

cfg = pd.read_csv("outputs/part_c_final_config.csv").iloc[0]
max_depth_text = str(cfg["max_depth"])
max_depth = None if max_depth_text.lower() == "unlimited" else int(float(max_depth_text))
rf_params = {"n_estimators": int(cfg["n_estimators"]), "max_depth": max_depth,
             "min_samples_leaf": int(cfg["min_samples_leaf"]), "random_state": 42}

# This definition fixes the earlier NameError: final_pipe is created here before joblib.dump.
final_pipe = Pipeline([("selector", SelectKBest(f_classif, k=int(cfg["selector_k"]))),
                       ("model", RandomForestClassifier(**rf_params))])
final_pipe.fit(X_train, y_train)

joblib.dump(final_pipe, "models/delay_model.pkl")
pd.DataFrame({"threshold": [float(cfg["threshold"])]}).to_csv("models/delay_threshold.csv", index=False)
pd.DataFrame({"feature": FEATURES}).to_csv("models/feature_order.csv", index=False)
pd.DataFrame({"column": list(medians.keys()), "median": list(medians.values())}).to_csv(
    "models/impute_values.csv", index=False)

reloaded = joblib.load("models/delay_model.pkl")
in_memory_prob = final_pipe.predict_proba(X_test)[:, 1]
reloaded_prob = reloaded.predict_proba(X_test)[:, 1]
diff = float(np.max(np.abs(in_memory_prob - reloaded_prob)))
pd.DataFrame({"metric": ["test_rows_scored", "reload_max_abs_probability_diff", "threshold"],
              "value": [len(X_test), diff, float(cfg["threshold"])]}).to_csv(
    "outputs/part_d_reload_check.csv", index=False, float_format="%.12f")
print("Model packaged. final_pipe fitted; reload difference:", diff)

ModuleNotFoundError: No module named 'src.shipment_prep'

**predict_delay.py**

In [65]:
"""Score new shipments and produce the deterministic operations action list."""
import pandas as pd
import joblib
from src.shipment_prep import clean_shipments, build_features

new_raw = pd.read_csv("data/new_shipments.csv")
hubs = pd.read_csv("data/hubs.csv")
model = joblib.load("models/delay_model.pkl")
threshold = float(pd.read_csv("models/delay_threshold.csv")["threshold"].iloc[0])
feature_order = pd.read_csv("models/feature_order.csv")["feature"].tolist()
impute = pd.read_csv("models/impute_values.csv")
medians = dict(zip(impute["column"], impute["median"].astype(float)))
hub_segments = pd.read_csv("outputs/hub_segments.csv")

clean_new, _ = clean_shipments(new_raw, medians=medians)
features = build_features(clean_new, hubs)
missing = [column for column in feature_order if column not in features.columns]
if missing:
    raise RuntimeError(f"Missing model features: {missing}")
probabilities = model.predict_proba(features[feature_order].astype(float))[:, 1]

out = features[["shipment_id", "hub_id"]].copy()
out["delay_probability"] = [round(float(p), 4) for p in probabilities]
out["flagged"] = (probabilities >= threshold).astype(int)
out = out.merge(hub_segments[["hub_id", "segment"]], on="hub_id", how="left", validate="many_to_one")
if out["segment"].isna().any():
    raise ValueError("A new shipment hub has no matching hub segment")

def choose_action(row):
    if row["flagged"] == 1 and row["segment"] == "Escalate":
        return "Escalate to hub manager"
    if row["flagged"] == 1:
        return "Proactive customer call"
    return "No action"

out["action"] = out.apply(choose_action, axis=1)
out = out[["shipment_id", "hub_id", "segment", "delay_probability", "flagged", "action"]]
out.to_csv("outputs/new_shipment_actions.csv", index=False)
print("Scored rows:", len(out), "Flagged:", int(out["flagged"].sum()))
print(out["action"].value_counts().to_string())

ModuleNotFoundError: No module named 'src.shipment_prep'

.

In [66]:
import os

print("Current directory:", os.getcwd())
print("\nFiles and folders:")

for item in os.listdir():
    print(item)

Current directory: /content

Files and folders:
.config
data
delhivery-delay-pipeline-v2
outputs
src
models
sample_data


In [67]:
import os

print("src folder exists:", os.path.isdir("src"))
print("shipment_prep.py exists:", os.path.isfile("src/shipment_prep.py"))

src folder exists: True
shipment_prep.py exists: False


In [68]:
import os

os.makedirs("src", exist_ok=True)

In [69]:
import shutil

if os.path.isfile("shipment_prep.py"):
    shutil.move("shipment_prep.py", "src/shipment_prep.py")

In [70]:
with open("src/__init__.py", "w") as f:
    pass

In [71]:
import os
import sys

PROJECT_DIR = os.getcwd()

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print("Project directory:", PROJECT_DIR)
print("Python import path updated.")

Project directory: /content
Python import path updated.


In [74]:
!pip install -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [75]:
!python part_a_clean_eda.py

python3: can't open file '/content/part_a_clean_eda.py': [Errno 2] No such file or directory


In [76]:
!python part_b_regression.py

python3: can't open file '/content/part_b_regression.py': [Errno 2] No such file or directory


In [77]:
!python part_c_classification.py

python3: can't open file '/content/part_c_classification.py': [Errno 2] No such file or directory


In [78]:
!python part_d_segments.py

python3: can't open file '/content/part_d_segments.py': [Errno 2] No such file or directory


In [79]:
!python package_model.py

python3: can't open file '/content/package_model.py': [Errno 2] No such file or directory


In [80]:
!python predict_delay.py

python3: can't open file '/content/predict_delay.py': [Errno 2] No such file or directory


In [73]:
import os

required_files = [
    "src/__init__.py",
    "src/shipment_prep.py",
    "part_a_clean_eda.py",
    "part_b_regression.py",
    "part_c_classification.py",
    "part_d_segments.py",
    "package_model.py",
    "predict_delay.py"
]

for file in required_files:
    status = "FOUND" if os.path.isfile(file) else "MISSING"
    print(f"{file}: {status}")

src/__init__.py: FOUND
src/shipment_prep.py: MISSING
part_a_clean_eda.py: MISSING
part_b_regression.py: MISSING
part_c_classification.py: MISSING
part_d_segments.py: MISSING
package_model.py: MISSING
predict_delay.py: MISSING
